# Lab 02 — Spark Data Processing
**SparkSession, DataFrame, filtering, sorting, null/casting, agregasi, pipeline dasar**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
Peserta membuat DataFrame, membaca schema, memakai `select/filter/orderBy`, membuat kolom turunan, menangani null, melakukan casting dan agregasi, lalu menyusun pipeline end-to-end.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

## Dataset bersama

In [ ]:
data = [
    (1, "Jakarta",  "Laptop",   2, 12000000),
    (2, "Bandung",  "Mouse",    5,   150000),
    (3, "Jakarta",  "Keyboard", 3,   500000),
    (4, "Surabaya", "Monitor",  2,  2500000),
    (5, "Bandung",  "Laptop",   1, 12000000),
    (6, "Jakarta",  "Mouse",   10,   150000),
]
columns = ["id", "city", "product", "qty", "price"]
df = spark.createDataFrame(data, columns)

## Lab 1 — Membuat dan mengenali DataFrame (15 menit)
Tampilkan seluruh data, schema, jumlah baris, nama kolom, dan jumlah partition. Catat tipe data `price`.

In [ ]:
df.show(truncate=False)
df.printSchema()
print("rows       :", df.count())
print("columns    :", df.columns)
print("partitions :", df.rdd.getNumPartitions())

## Lab 2 — Select, filter, sorting (20 menit)
**Tugas:** tampilkan transaksi Jakarta dengan harga satuan minimal Rp500.000. Ambil `product`, `qty`, `price`, urutkan harga terbesar.

In [ ]:
# TODO peserta: tulis pipeline di bawah.
lab2 = (
    df
    # .filter(...)
    # .select(...)
    # .orderBy(...)
)
# lab2.show(truncate=False)

In [ ]:
# Reference solution
lab2_solution = (
    df
    .filter((F.col("city") == "Jakarta") & (F.col("price") >= 500_000))
    .select("product", "qty", "price")
    .orderBy(F.col("price").desc())
)
lab2_solution.show(truncate=False)

## Lab 3 — Data quality, null, casting, kategori (30 menit)

In [ ]:
dirty = spark.createDataFrame([
    (101, " Jakarta ", "Laptop",   "2", "12000000", "VALID"),
    (102, "Bandung",   "Mouse",    "5", "150000",   "VALID"),
    (103, "Jakarta",   None,       "3", "500000",   "VALID"),
    (104, None,         "Monitor",  "2", "2500000",  "CANCEL"),
    (105, "Bandung",   "Laptop",   None, "12000000", "VALID"),
], ["transaction_id", "city", "product", "qty_raw", "price_raw", "status"])

dirty.show(truncate=False)
dirty.printSchema()

**Tugas:**
1. `trim()` city.
2. Cast `qty_raw` dan `price_raw` ke `long`.
3. Buang baris `product IS NULL`.
4. Isi qty null menjadi 0.
5. Buat `total_value = qty * price`.
6. Kategori HIGH ≥ 10 juta, MEDIUM ≥ 1 juta, selain itu LOW.

In [ ]:
# TODO peserta
cleaned = dirty
# lanjutkan transformasi...

In [ ]:
cleaned_solution = (
    dirty
    .withColumn("city", F.trim("city"))
    .withColumn("qty", F.col("qty_raw").cast("long"))
    .withColumn("price", F.col("price_raw").cast("long"))
    .filter(F.col("product").isNotNull())
    .na.fill({"qty": 0, "city": "UNKNOWN"})
    .withColumn("total_value", F.col("qty") * F.col("price"))
    .withColumn(
        "category",
        F.when(F.col("total_value") >= 10_000_000, "HIGH")
         .when(F.col("total_value") >= 1_000_000, "MEDIUM")
         .otherwise("LOW")
    )
    .select("transaction_id", "city", "product", "qty", "price", "status", "total_value", "category")
)
cleaned_solution.show(truncate=False)
cleaned_solution.printSchema()

## Lab 4 — Agregasi per kota (25 menit)
Hitung total quantity dan total nilai transaksi per kota dari dataset awal. Urutkan berdasarkan total nilai terbesar. Setelah itu jalankan `explain("formatted")`.

In [ ]:
df_total = df.withColumn("total", F.col("qty") * F.col("price"))
summary = (
    df_total
    .groupBy("city")
    .agg(
        F.sum("qty").alias("total_qty"),
        F.sum("total").alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
)
summary.show(truncate=False)
summary.explain("formatted")

**Validasi manual:** Jakarta = 27.000.000; Bandung = 12.750.000; Surabaya = 5.000.000. Cari `Exchange` dan jelaskan mengapa agregasi berpotensi shuffle.

## Lab 5 — Mini case end-to-end (35 menit)

In [ ]:
transactions = [
    (101, "Jakarta", "Laptop", 2, 12000000, "VALID"),
    (102, "Bandung", "Mouse", 5, 150000, "VALID"),
    (103, "Jakarta", None, 3, 500000, "VALID"),
    (104, "Surabaya", "Monitor", 2, 2500000, "CANCEL"),
    (105, "Bandung", "Laptop", 1, 12000000, "VALID"),
    (106, "Jakarta", "Mouse", 10, 150000, "VALID"),
]
cols = ["transaction_id", "city", "product", "qty", "price", "status"]
raw_df = spark.createDataFrame(transactions, cols)

**Tugas peserta:** ambil `VALID`; buang product null; hitung `total_value`; buat category; pilih kolom relevan; urutkan nilai terbesar; validasi count dan satu perhitungan manual.

In [ ]:
# TODO peserta
processed_df = raw_df

In [ ]:
processed_solution = (
    raw_df
    .filter(F.col("status") == "VALID")
    .filter(F.col("product").isNotNull())
    .withColumn("total_value", F.col("qty") * F.col("price"))
    .withColumn(
        "category",
        F.when(F.col("total_value") >= 10_000_000, "HIGH")
         .when(F.col("total_value") >= 1_000_000, "MEDIUM")
         .otherwise("LOW")
    )
    .select("transaction_id", "city", "product", "total_value", "category")
    .orderBy(F.col("total_value").desc())
)
processed_solution.show(truncate=False)
print("rows:", processed_solution.count())
processed_solution.explain("formatted")

## Exit ticket
Mengapa loop Python per-row dan `collect()` pada data besar bertentangan dengan mental model Spark? Kapan `show()`/`limit()` lebih aman?